# MobileNetV2 Regression on Girasol IR images

MobileNetV2 trained from scratch (random initialisation, no pretrained weights) with a linear regression head to predict solar irradiance.

In [1]:
# Standard Libraries
import os
import math
import time
import random
import pickle
import cv2
import numpy as np
import pandas as pd

# External Libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

from tqdm.auto import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# import scripts
from scripts.saveResults import saveLossDiagram

start = time.time()

/home/ifran/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
# Data protocol (same as Train/EfficientNetB0/EfficientNetConv/efficientnet_conv_cuda.py)
DATASET_CSV = "/home/ifran/Projects_UBUNTU/Girasol_EfficientNet_Regression/girasol.csv"
MEAN = [0.3938, 0.3938, 0.3938]
STD = [0.0242, 0.0242, 0.0242]
TRAIN_RATIO = 0.7
RANDOM_STATE = 42
BATCH_SIZE = 256
NUM_WORKERS = 4

# Training hyperparameters
NUM_EPOCHS = 50
PATIENCE = 5  # early stopping: epochs without validation RMSE improvement
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
LR_FACTOR = 0.1  # multiply LR by this when validation RMSE plateaus
LR_PATIENCE = 2  # epochs without improvement before reducing LR

RESULT_DIR = "result/mobilenet_regression_v2/"

## Dataset

In [3]:
class SolarRadiance(Dataset):
    """CSV rows of (image path, irradiance). Images are resized to 80x60 (W x H)."""

    def __init__(self, labels, transform):
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        img_path = self.labels.iloc[index, 0]
        target = self.labels.iloc[index, 1]

        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        image = cv2.resize(image, (80, 60))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(target, dtype=torch.float32)


def train_val_split(csv_path):
    """Chronological split: first 70% of rows for training, remaining 30% for validation."""
    data = pd.read_csv(csv_path)
    train_size = int(TRAIN_RATIO * len(data))
    return data[:train_size], data[train_size:]

## Training and evaluation

In [4]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [train]"):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()
    predictions, labels = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Evaluating"):
            images = images.to(device, non_blocking=True)
            outputs = model(images).squeeze(1)
            predictions.append(outputs.cpu().numpy())
            labels.append(targets.numpy())
    predictions = np.concatenate(predictions)
    labels = np.concatenate(labels)

    # Back to W/m^2 before computing metrics
    predictions = predictions * (TARGET_MAX - TARGET_MIN) + TARGET_MIN
    labels = labels * (TARGET_MAX - TARGET_MIN) + TARGET_MIN

    mse = mean_squared_error(labels, predictions)
    return {
        "MSE": mse,
        "RMSE": math.sqrt(mse),
        "MAE": mean_absolute_error(labels, predictions),
        "R2": r2_score(labels, predictions),
    }

## Setup

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)


def seed_everything(seed):
    """Seed all RNGs and force deterministic cuDNN kernels."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    """Give each DataLoader worker a reproducible seed derived from the base seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(RANDOM_STATE)
g = torch.Generator()
g.manual_seed(RANDOM_STATE)

train_data, val_data = train_val_split(DATASET_CSV)
print(f"Train samples: {len(train_data)} | Validation samples: {len(val_data)}")

# Min-max scale the irradiance targets with training-split statistics; metrics are reported in W/m^2
TARGET_MIN = train_data.iloc[:, 1].min()
TARGET_MAX = train_data.iloc[:, 1].max()
print(f"Target scaling: min {TARGET_MIN:.2f}, max {TARGET_MAX:.2f} W/m^2")
train_data, val_data = train_data.copy(), val_data.copy()
train_data.iloc[:, 1] = (train_data.iloc[:, 1] - TARGET_MIN) / (TARGET_MAX - TARGET_MIN)
val_data.iloc[:, 1] = (val_data.iloc[:, 1] - TARGET_MIN) / (TARGET_MAX - TARGET_MIN)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_loader = DataLoader(SolarRadiance(train_data, transform), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                          worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(SolarRadiance(val_data, transform), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                        worker_init_fn=seed_worker, generator=g)

Device:  cuda


Train samples: 348785 | Validation samples: 149480
Target scaling: min 9.27, max 1271.67 W/m^2


## Model

In [6]:
# MobileNetV2 with random initialisation (no pretrained weights)
mobilenet = models.mobilenet_v2(pretrained=False)
# Remove the last layer (classification layer)
mobilenet = nn.Sequential(*list(mobilenet.children())[:-1])

# Add custom layers for regression
class LinearRegressionHead(nn.Module):
    def __init__(self, in_features):
        super(LinearRegressionHead, self).__init__()
        self.fc = nn.Linear(in_features, 1)  # Output 1 value for linear regression
    
    def forward(self, x):
        x = self.fc(x)
        return x

# Append the custom regression head to the MobileNetV2 backbone
in_features = mobilenet[-1][-1].out_channels  # Get the number of output channels from the last layer
model = nn.Sequential(
    mobilenet,
    nn.AdaptiveAvgPool2d(1),  
    nn.Flatten(),
    LinearRegressionHead(in_features)
)
model.to(device)

/home/ifran/miniconda3/envs/torch/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ifran/miniconda3/envs/torch/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1

## Loss, optimizer and checkpoint paths

In [7]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE)

os.makedirs(RESULT_DIR, exist_ok=True)
best_path = os.path.join(RESULT_DIR, "mobilenet_regression.pth")
losses_path = os.path.join(RESULT_DIR, "losses.pkl")

## Train

In [8]:
best_rmse = float("inf")
epochs_no_improve = 0
losses = []  # validation MSE per epoch
for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    metrics = evaluate(model, val_loader, device)
    scheduler.step(metrics["RMSE"])
    losses.append(metrics["MSE"])

    print(f"Epoch {epoch + 1}: train MSE {train_loss:.4f} | "
          f"val MSE {metrics['MSE']:.4f} RMSE {metrics['RMSE']:.4f} "
          f"MAE {metrics['MAE']:.4f} R2 {metrics['R2']:.4f} | "
          f"LR {optimizer.param_groups[0]['lr']:.1e}")

    if metrics["RMSE"] < best_rmse:
        best_rmse = metrics["RMSE"]
        torch.save(model.state_dict(), best_path)
        print(f"Saved best model to {best_path}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping: no improvement for {PATIENCE} epochs")
            break

# Save the validation MSE per epoch and its curve (model_loss.png)
with open(losses_path, "wb") as file:
    pickle.dump(losses, file)
saveLossDiagram(len(losses), losses, RESULT_DIR)

Epoch 1/50 [train]:   0%|          | 0/1363 [00:00<?, ?it/s]

Epoch 1/50 [train]:   0%|          | 1/1363 [00:03<1:27:19,  3.85s/it]

Epoch 1/50 [train]:   0%|          | 2/1363 [00:04<38:51,  1.71s/it]  

Epoch 1/50 [train]:   0%|          | 3/1363 [00:04<22:44,  1.00s/it]

Epoch 1/50 [train]:   0%|          | 4/1363 [00:04<15:05,  1.50it/s]

Epoch 1/50 [train]:   0%|          | 5/1363 [00:04<11:23,  1.99it/s]

... (39,909 outputs left out)

Evaluating: 100%|██████████| 584/584 [00:59<00:00, 16.20it/s]

Evaluating: 100%|██████████| 584/584 [00:59<00:00,  9.76it/s]

Epoch 25: train MSE 0.0002 | val MSE 4376.8221 RMSE 66.1576 MAE 40.0922 R2 0.9248 | LR 1.0e-06
Early stopping: no improvement for 5 epochs


/home/ifran/Projects_UBUNTU/Girasol_EfficientNet_Regression/Thesis_Chapter_7/MobilenetRegression/scripts/saveResults.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final evaluation (best checkpoint on validation set)

In [9]:
# Final evaluation with the best checkpoint
model.load_state_dict(torch.load(best_path, map_location=device))
metrics = evaluate(model, val_loader, device)
print("************ MobileNetV2 Regression ************")
print("MSE:", metrics["MSE"])
print("RMSE:", metrics["RMSE"])
print("MAE:", metrics["MAE"])
print("R2 Score", metrics["R2"])

total_time = time.time() - start
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"Run Time: {minutes} minutes and {seconds} seconds")

Evaluating:   0%|          | 0/584 [00:00<?, ?it/s]

Evaluating:   0%|          | 1/584 [00:01<14:24,  1.48s/it]

Evaluating:   1%|          | 3/584 [00:01<04:13,  2.30it/s]

Evaluating:   1%|          | 4/584 [00:01<03:08,  3.07it/s]

Evaluating:   1%|          | 6/584 [00:01<01:55,  5.01it/s]

Evaluating:   1%|▏         | 8/584 [00:02<01:23,  6.89it/s]

... (282 outputs left out)

Evaluating:  99%|█████████▉| 581/584 [00:45<00:00, 15.75it/s]

Evaluating: 100%|█████████▉| 583/584 [00:45<00:00, 15.77it/s]

Evaluating: 100%|██████████| 584/584 [00:45<00:00, 12.73it/s]

************ MobileNetV2 Regression ************
MSE: 4294.507883047876
RMSE: 65.53249486360089
MAE: 39.924212069515626
R2 Score 0.9261689801937402
Run Time: 132 minutes and 49 seconds
